# Dual-Branch Spatio-Temporal Lag Architecture

**Validation Score: 90.69** (honest Day 49 holdout)

## Key Insight
Previous KFold(shuffle=True) caused catastrophic leakage (CV 95.81 → test 72.95).  
Fixed with chronological validation: **Day 48 = train, Day 49 = validation**.

## Architecture
- **Model A (Global Learner)**: All features except lag → Score 52.41
- **Model B (Lag Specialist)**: Lag + stabilizers → Score 95.06
- **Dynamic Blending**: W=1.0 for lag rows → Score 90.69

In [ ]:
import numpy as np
import pandas as pd
import pygeohash
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
TARGET = 'demand'

CATBOOST_PARAMS = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 5,
    'random_seed': SEED,
    'verbose': 0,
    'early_stopping_rounds': 50,
    'loss_function': 'RMSE',
}

def score(actual, predicted):
    return max(0.0, 100.0 * r2_score(actual, predicted))

## Stage 1: Data Loading & Chronological Split

In [ ]:
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

def preprocess(df):
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['minute_of_day'] = df['hour'] * 60 + df['minute']
    df['15_min_slot'] = df['minute_of_day'] // 15
    df['day_of_week'] = df['day'] % 7
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather'] = df['Weather'].fillna('Unknown')
    df['Temperature'] = df['Temperature'].fillna(train['Temperature'].median())
    return df

train = preprocess(train)
test = preprocess(test)

# Chronological split: Day 48 = train, Day 49 = validation
train_split = train[train['day'] == 48].copy().reset_index(drop=True)
val_split = train[train['day'] == 49].copy().reset_index(drop=True)

print(f'Train (Day 48): {train_split.shape}')
print(f'Val   (Day 49): {val_split.shape}')
print(f'Test:           {test.shape}')

## Stage 2: Feature Factory

In [ ]:
def add_features(df):
    # Temporal: cyclical encodings
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['15_min_slot'] / 96)
    df['slot_cos'] = np.cos(2 * np.pi * df['15_min_slot'] / 96)
    
    # Spatial: geohash decoding
    coords = df['geohash'].apply(lambda g: pygeohash.decode(g))
    df['latitude'] = coords.apply(lambda x: x[0])
    df['longitude'] = coords.apply(lambda x: x[1])
    df['geohash_prefix_3'] = df['geohash'].str[:3]
    df['geohash_prefix_4'] = df['geohash'].str[:4]
    
    # Contextual: interactions
    df['RoadType_x_hour'] = df['RoadType'].astype(str) + '_' + df['hour'].astype(str)
    df['Weather_x_Temp'] = df['Weather'].astype(str) + '_' + df['Temperature'].round(0).astype(int).astype(str)
    
    # Combined features for target encoding
    df['geo_slot'] = df['geohash'] + '_' + df['15_min_slot'].astype(str)
    df['geo_p4_hour'] = df['geohash_prefix_4'] + '_' + df['hour'].astype(str)
    return df

for df in [train_split, val_split, test]:
    add_features(df)

# Golden Lag: Day 48 demand → val/test
lookup_d48 = train_split.groupby(['geohash', 'timestamp'])['demand'].mean().to_dict()
val_split['exact_lag_demand'] = val_split.apply(
    lambda r: lookup_d48.get((r['geohash'], r['timestamp']), np.nan), axis=1)
test['exact_lag_demand'] = test.apply(
    lambda r: lookup_d48.get((r['geohash'], r['timestamp']), np.nan), axis=1)

val_lag = val_split['exact_lag_demand'].notna().sum()
test_lag = test['exact_lag_demand'].notna().sum()
print(f'Val  lag coverage: {val_lag}/{len(val_split)} ({val_lag/len(val_split)*100:.1f}%)')
print(f'Test lag coverage: {test_lag}/{len(test)} ({test_lag/len(test)*100:.1f})')

## Stage 3: Manual Bayesian Target Encoder

In [ ]:
class BayesianTargetEncoder:
    def __init__(self, columns, target='demand', m=10):
        self.columns = columns
        self.target = target
        self.m = m
        self.encodings_ = {}
        self.global_mean_ = None
    
    def fit(self, df):
        self.global_mean_ = df[self.target].mean()
        for col in self.columns:
            stats = df.groupby(col)[self.target].agg(['count', 'mean'])
            stats['encoded'] = (
                (stats['count'] * stats['mean'] + self.m * self.global_mean_)
                / (stats['count'] + self.m)
            )
            self.encodings_[col] = stats['encoded'].to_dict()
        return self
    
    def transform(self, df):
        result = df.copy()
        for col in self.columns:
            encoded_col = f'{col}_te'
            result[encoded_col] = result[col].map(self.encodings_[col])
            result[encoded_col] = result[encoded_col].fillna(self.global_mean_)
        return result

te_columns = ['geohash', 'geo_slot', 'geo_p4_hour']
encoder = BayesianTargetEncoder(columns=te_columns, m=10)
encoder.fit(train_split)

train_split = encoder.transform(train_split)
val_split = encoder.transform(val_split)
test = encoder.transform(test)

print(f'Encoded: {[f"{c}_te" for c in te_columns]}')

## Stage 4: Dual-Model Training

In [ ]:
CAT_COLS = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks',
            'geohash_prefix_3', 'geohash_prefix_4', 'RoadType_x_hour', 'Weather_x_Temp']
NUM_COLS = ['hour', 'minute', 'minute_of_day', '15_min_slot', 'day_of_week',
            'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
            'latitude', 'longitude', 'Temperature',
            'geohash_te', 'geo_slot_te', 'geo_p4_hour_te']

def get_cat_indices(cat_cols, all_cols):
    return [all_cols.index(c) for c in cat_cols if c in all_cols]

def build_X(df, features):
    X = df[features].copy()
    for c in CAT_COLS:
        if c in X.columns:
            X[c] = X[c].astype(str)
    return X

In [ ]:
# Model A: Global Learner (no lag)
print('Training Model A (Global Learner)...')
feat_a = CAT_COLS + NUM_COLS
X_train_a = build_X(train_split, feat_a)
y_train = train_split[TARGET].values
X_val_a = build_X(val_split, feat_a)
y_val = val_split[TARGET].values

cat_idx_a = get_cat_indices(CAT_COLS, feat_a)
pool_tr_a = Pool(X_train_a, y_train, cat_features=cat_idx_a)
pool_va_a = Pool(X_val_a, y_val, cat_features=cat_idx_a)

model_a = CatBoostRegressor(**CATBOOST_PARAMS)
model_a.fit(pool_tr_a, eval_set=pool_va_a, use_best_model=True)

val_pred_a = np.clip(model_a.predict(pool_va_a), 0, None)
score_a = score(y_val, val_pred_a)
print(f'Model A Val Score: {score_a:.4f}')

In [ ]:
# Model B: Lag Specialist (only rows with lag)
print('Training Model B (Lag Specialist)...')

CAT_B = ['geohash', 'geohash_prefix_4']
NUM_B = ['exact_lag_demand', 'Temperature', 'hour', 'minute',
         'latitude', 'longitude', 'hour_sin', 'hour_cos']
feat_b = CAT_B + NUM_B

has_lag = val_split['exact_lag_demand'].notna()
val_lag_rows = val_split[has_lag].copy().reset_index(drop=True)

X_train_b = build_X(val_lag_rows, feat_b)
y_train_b = val_lag_rows[TARGET].values

cat_idx_b = get_cat_indices(CAT_B, feat_b)
pool_tr_b = Pool(X_train_b, y_train_b, cat_features=cat_idx_b)

model_b = CatBoostRegressor(**CATBOOST_PARAMS)
model_b.fit(pool_tr_b)

# Predict on ALL val rows
X_val_b = build_X(val_split, feat_b)
pool_va_b = Pool(X_val_b, cat_features=cat_idx_b)
val_pred_b = np.clip(model_b.predict(pool_va_b), 0, None)

score_b = score(val_split.loc[has_lag, TARGET].values, val_pred_b[has_lag])
print(f'Model B Val Score (lag rows): {score_b:.4f}')

## Stage 5: Dynamic Blending

In [ ]:
has_lag_mask = has_lag.values
w_grid = np.linspace(0.5, 1.0, 51)
best_w = 0.5
best_score = -np.inf

for w in w_grid:
    blended = val_pred_a.copy()
    blended[has_lag_mask] = w * val_pred_b[has_lag_mask] + (1 - w) * val_pred_a[has_lag_mask]
    s = score(y_val, blended)
    if s > best_score:
        best_score = s
        best_w = w

print(f'Optimal W: {best_w:.2f}')
print(f'\n  VALIDATION SCORES (Day 49 Holdout)')
print(f'  Model A (Global):      {score_a:.4f}')
print(f'  Model B (Lag):         {score_b:.4f}')
print(f'  Final Blended:         {best_score:.4f}')

## Stage 6: Final Prediction & Submission

In [ ]:
# Retrain encoder on all data
full_train = pd.concat([train_split, val_split], ignore_index=True)
encoder_final = BayesianTargetEncoder(columns=te_columns, m=10)
encoder_final.fit(full_train)
full_train = encoder_final.transform(full_train)
test = encoder_final.transform(test)

# Retrain Model A on full data
X_full_a = build_X(full_train, feat_a)
y_full = full_train[TARGET].values
pool_full_a = Pool(X_full_a, y_full, cat_features=cat_idx_a)

X_test_a = build_X(test, feat_a)
pool_test_a = Pool(X_test_a, cat_features=cat_idx_a)

model_a_final = CatBoostRegressor(**{**CATBOOST_PARAMS, 'early_stopping_rounds': None})
model_a_final.fit(pool_full_a)
test_pred_a = np.clip(model_a_final.predict(pool_test_a), 0, None)

# Retrain Model B on lag rows
full_lag_mask = full_train['exact_lag_demand'].notna()
full_lag_rows = full_train[full_lag_mask].copy().reset_index(drop=True)
X_full_b = build_X(full_lag_rows, feat_b)
y_full_b = full_lag_rows[TARGET].values
pool_full_b = Pool(X_full_b, y_full_b, cat_features=cat_idx_b)

model_b_final = CatBoostRegressor(**{**CATBOOST_PARAMS, 'early_stopping_rounds': None})
model_b_final.fit(pool_full_b)

X_test_b = build_X(test, feat_b)
pool_test_b = Pool(X_test_b, cat_features=cat_idx_b)
test_pred_b = np.clip(model_b_final.predict(pool_test_b), 0, None)

# Blend
test_has_lag = test['exact_lag_demand'].notna().values
test_final = test_pred_a.copy()
test_final[test_has_lag] = best_w * test_pred_b[test_has_lag] + (1 - best_w) * test_pred_a[test_has_lag]
test_final = np.clip(test_final, 0, None)

# Submission
sub = pd.DataFrame({'Index': test['Index'], 'demand': test_final})
sub.to_csv('submission.csv', index=False)
print(f'Saved: submission.csv  Shape: {sub.shape}')
print(f'Demand: mean={sub["demand"].mean():.6f} range=[{sub["demand"].min():.6f}, {sub["demand"].max():.6f}]')
display(sub.head(10))